# Phase 3 — Model L (Nepali): reasoning finetuning and analysis

The Nepali counterpart of `finetune_hindi_colab.ipynb`. Same code, same
hyperparameters, different language — that is the point: Model H and Model L
differ in their data, never in their treatment.

Kaggle has no `drive.mount`, so **rclone** carries the pretrained checkpoint in
and every finetuned checkpoint back out. `/kaggle/working` is discarded when an
interactive session dies, so an un-mirrored run loses everything on a
disconnect.

### Before running

1. **Settings -> Internet: ON** (required for rclone)
2. **Settings -> Accelerator: GPU T4**
3. **Add-ons -> Secrets**, label `RCLONE_CONF`, paste `rclone config show gdrive:`
   from your machine, including the `[gdrive]` header line.

> That config holds an OAuth refresh token for your Drive. Kaggle Secrets are
> encrypted and scoped to your own sessions, but revoke the token afterwards if
> the notebook is ever shared.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Settings -> Accelerator -> GPU T4'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

## 2. rclone

In [ ]:
import os, re
from kaggle_secrets import UserSecretsClient

!curl -s https://rclone.org/install.sh | bash > /dev/null 2>&1


def normalize_rclone_conf(conf: str) -> str:
    """Restore an rclone config whose newlines were flattened to spaces.

    Kaggle's Secrets UI stores a multi-line value on a single line, but rclone
    needs real line breaks or it reports 'didn't find section in config file'.
    Re-insert a break before each [section] and each key=. Safe because rclone
    writes the OAuth token as compact JSON containing no spaces.
    """
    if len(conf.strip().splitlines()) > 1:
        return conf if conf.endswith('\n') else conf + '\n'
    s = conf.strip().replace('\\n', '\n')
    if len(s.splitlines()) > 1:
        return s + '\n'
    s = re.sub(r'\s*(\[[^\]\s]+\])\s*', r'\n\1\n', s)
    s = re.sub(r'\s+([A-Za-z_][A-Za-z0-9_]*)\s*=\s*', r'\n\1 = ', s)
    return '\n'.join(l.strip() for l in s.splitlines() if l.strip()) + '\n'


conf = normalize_rclone_conf(UserSecretsClient().get_secret('RCLONE_CONF'))

# RCLONE_CONFIG names the file explicitly: Kaggle's $HOME is not reliably /root,
# and the variable is inherited by every subprocess, including the checkpoint
# mirroring inside trainer.py.
CONF_PATH = '/kaggle/working/rclone.conf'
with open(CONF_PATH, 'w') as fh:
    fh.write(conf)
os.environ['RCLONE_CONFIG'] = CONF_PATH

lines = conf.splitlines()
print('lines        :', len(lines))
print('has [gdrive] :', any(l.strip() == '[gdrive]' for l in lines))
print('keys         :', [l.split('=')[0].strip() for l in lines if '=' in l])
assert any(l.strip() == '[gdrive]' for l in lines), 'RCLONE_CONF has no [gdrive] section'

!rclone listremotes
!rclone lsd gdrive:24KaratAu

## 3. Code and the pretrained checkpoint

The Phase 2 checkpoint is pulled into `/kaggle/working/pretrained/`, which is
where `nepali/configs/finetune.yaml` expects it.

In [ ]:
DRIVE = 'gdrive:24KaratAu'

!rclone copy {DRIVE}/code /kaggle/working/phase3 --exclude "__pycache__/**"
%cd /kaggle/working/phase3
!pip install -q pyyaml sentencepiece

!mkdir -p /kaggle/working/pretrained
!rclone copy {DRIVE}/nepali/checkpoints/checkpoint_best.pt \
    /kaggle/working/pretrained -P --stats 10s
!ls -la /kaggle/working/pretrained

## 4. Build the reasoning corpus

Generated here, not copied, so the leakage checks run on the exact bytes that
get trained on. The Nepali inventories, phrasings and grammar in
`src/reasoning/entities.py` are written independently of the Hindi ones — the
two corpora share the generator and nothing else.

In [ ]:
!python -u -m src.reasoning.generate --language nepali \
    --out-dir data/nepali/reasoning \
    --train 40000 --val 5000 --test 5000 --seed 42

In [ ]:
import json
stats = json.load(open('data/nepali/reasoning/stats.json'))
print(json.dumps(stats['leakage'], ensure_ascii=False, indent=2))
assert not stats['leakage']['train_test_shared_entities']
assert stats['leakage']['train_test_shared_prompts'] == 0
assert stats['leakage']['val_test_shared_prompts'] == 0
print('tokenizer:', json.dumps(stats['tokenizer']['train'], ensure_ascii=False))

In [ ]:
!python -m pytest tests/ -q

In [ ]:
!mkdir -p /kaggle/working/reasoning_archive
!cp data/nepali/reasoning/* /kaggle/working/reasoning_archive/
!rclone copy /kaggle/working/reasoning_archive {DRIVE}/nepali/reasoning \
    -P --stats 15s

## 5. Finetune (~6 min per run)

Checkpoints are mirrored to Drive by the trainer itself after every save, so a
session death costs at most `save_latest_every` steps rather than the run.

In [ ]:
import os, yaml
cfg = yaml.safe_load(open('nepali/configs/finetune.yaml'))
assert os.path.exists(cfg['init_from']), 'pretrained checkpoint not copied in'
print('init from  :', cfg['init_from'])
print('mirror to  :', cfg['checkpoint']['mirror_to'])

In [ ]:
!python -u -m src.train.finetune --config nepali/configs/finetune.yaml

In [ ]:
!python -u -m src.train.finetune --config nepali/configs/finetune_rationale.yaml

## 6. Pretrained vs finetuned exact match

In [ ]:
PRE  = '/kaggle/working/pretrained/checkpoint_best.pt'
FT_D = '/kaggle/working/checkpoints_ft_direct/finetune_best.pt'
FT_R = '/kaggle/working/checkpoints_ft_rationale/finetune_best.pt'
COMMON = ('--model-config nepali/configs/model.yaml '
          '--tokenizer tokenizers/nepali/nepali_bpe_8k.model '
          '--test data/nepali/reasoning/test.jsonl --language nepali')
os.makedirs('nepali/eval', exist_ok=True)

In [ ]:
!python -u -m src.eval.reasoning_eval --checkpoint {PRE} {COMMON} \
    --mode zeroshot --answer-style direct \
    --out nepali/eval/reasoning_zeroshot_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {PRE} {COMMON} \
    --mode fewshot --shots 4 --train data/nepali/reasoning/train.jsonl \
    --answer-style direct --out nepali/eval/reasoning_fewshot_direct.json

In [ ]:
!python -u -m src.eval.reasoning_eval --checkpoint {FT_D} {COMMON} \
    --mode finetuned --answer-style direct \
    --out nepali/eval/reasoning_finetuned_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {FT_R} {COMMON} \
    --mode finetuned --answer-style rationale \
    --out nepali/eval/reasoning_finetuned_rationale.json

## 7. Attention: pretrained vs finetuned

In [ ]:
!python -u -m src.eval.attention_reasoning \
    --pretrained {PRE} --finetuned {FT_D} \
    --model-config nepali/configs/model.yaml \
    --tokenizer tokenizers/nepali/nepali_bpe_8k.model \
    --test data/nepali/reasoning/test.jsonl --language nepali --tag direct \
    --outdir report/figures --out nepali/eval/attention_reasoning.json

## 8. Push everything off the box

`/kaggle/working` does not survive the session. Nothing below is optional.

In [ ]:
!rclone copy nepali/eval {DRIVE}/nepali/phase3/eval -P
!rclone copy report/figures {DRIVE}/nepali/phase3/figures -P
!rclone copy /kaggle/working/logs {DRIVE}/nepali/logs -P
!rclone ls {DRIVE}/nepali/phase3 | head -30
print('\nPull these into the repo and commit: nepali/eval/*.json,')
print('nepali/logs/finetune_*.jsonl, report/figures/*,')
print('data/nepali/reasoning/{stats.json,sample.md}')